# Building AI Agents with Databricks

Learn how to build production-ready AI agents using Databricks Foundation Model APIs. This notebook demonstrates a step-by-step approach from basic LLM calls to intelligent agents that autonomously use tools.

## What we about to do
1. **Foundation Model Integration** - Connect to Databricks-hosted LLMs (Claude 3.7 Sonnet, Llama 3.3 70B)
2. **Tool-Using Agents** - Build agents that decide when to execute Python code
3. **Agentic Workflows** - Implement autonomous reasoning and action loops

## Prerequisites 
* databricks community free edition

* Databricks workspace with Foundation Model APIs enabled
* Access to Claude 3.7 Sonnet or Llama 3.3 70B Instruct

## Step 1: Environment Setup

Install the required libraries:

* **mlflow** - Experiment tracking and LLM observability
* **databricks-openai** - OpenAI-compatible client for Databricks Foundation Model APIs
* **databricks-agents** - Agent development SDK with Unity Catalog function integration

In [0]:
%pip install mlflow databricks-openai databricks-agents -q

In [0]:
dbutils.library.restartPython()

## Step 2: Configure Foundation Model Endpoint

Automatically detect and configure the available LLM endpoint:

1. Test connectivity to Databricks Foundation Model APIs
2. Select between Claude 3.7 Sonnet or Llama 3.3 70B Instruct
3. Validate endpoint availability

In [0]:
# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [0]:
from databricks.sdk import WorkspaceClient

# Initialize LLM endpoint name
LLM_ENDPOINT_NAME = None

def is_endpoint_available(endpoint_name):
    """Test if a specific LLM endpoint is available and responding."""
    try:
        client = WorkspaceClient().serving_endpoints.get_open_ai_client()
        client.chat.completions.create(
            model=endpoint_name, 
            messages=[{"role": "user", "content": "What is AI?"}]
        )
        return True
    except Exception:
        return False

# Try candidate endpoints in order of preference
client = WorkspaceClient()
for candidate_endpoint_name in [
    "databricks-claude-3-7-sonnet", 
    "databricks-meta-llama-3-3-70b-instruct"
]:
    if is_endpoint_available(candidate_endpoint_name):
        LLM_ENDPOINT_NAME = candidate_endpoint_name
        break

assert LLM_ENDPOINT_NAME is not None, "No available LLM endpoint found"

# Display the selected endpoint
print(f"Using LLM endpoint: {LLM_ENDPOINT_NAME}")
LLM_ENDPOINT_NAME

## Step 3: Basic LLM Interaction

Make your first call to the Foundation Model API using the OpenAI-compatible interface.

**Key Components:**
* **MLflow autolog** - Automatic tracing and observability for all LLM calls
* **Chat Completions API** - Industry-standard interface for conversational AI
* **Databricks Model Serving** - Fully managed inference infrastructure

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient

# Automatically log traces from LLM calls for ease of debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to talk to Databricks model serving endpoints
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Define a test prompt
prompt = "what countries 2026 Fifa world cup play in?"

# Make the LLM call
response = openai_client.chat.completions.create(
    model=LLM_ENDPOINT_NAME,
    messages=[{"role": "user", "content": prompt}],
)

# Extract and display the response
response_text = response.choices[0].message.content
print(f"Prompt: {prompt}\n")
print(f"Response: {response_text}")

## Step 4: Create a Reusable LLM Function

Encapsulate the LLM interaction pattern into a reusable function for cleaner code architecture.

This abstraction:
* Standardizes prompt-response patterns
* Returns structured message objects
* Enables easy integration into larger systems

In [0]:
def run_llm(prompt):
    """
    Send a user prompt to the LLM and return response messages.
    
    Args:
        prompt (str): The user's question or instruction
        
    Returns:
        list: List of response message dictionaries
    """
    result_msgs = []
    
    # Send the prompt to the LLM endpoint
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=[{"role": "user", "content": prompt}],
    )
    
    # Extract and store the message
    msg = response.choices[0].message
    result_msgs.append(msg.to_dict())
    
    return result_msgs

## Step 5: Build an Agentic System with Tool Use

Create an intelligent agent that autonomously uses the Python code interpreter when needed.

**Agentic Capabilities:**
* **Autonomous reasoning** - Decides when computation is required
* **Tool execution** - Calls `system.ai.python_exec` to run Python code
* **Answer synthesis** - Interprets results and formulates responses

This demonstrates the core agentic pattern: reasoning → action → observation → answer.

In [0]:
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient

# Load the Python code interpreter tool
tools = UCFunctionToolkit(
    function_names=["system.ai.python_exec"],
    client=DatabricksFunctionClient()
).tools

def run_agent(prompt):
    """Run an agent that can use Python code interpreter."""
    messages = [{"role": "user", "content": prompt}]
    
    # Call LLM with tools
    response = openai_client.chat.completions.create(
        model=LLM_ENDPOINT_NAME,
        messages=messages,
        tools=tools
    )
    
    msg = response.choices[0].message
    messages.append(msg.to_dict())
    
    # Execute tool if called
    if msg.tool_calls:
        call = msg.tool_calls[0]
        result = DatabricksFunctionClient().execute_function(
            "system.ai.python_exec",
            parameters=json.loads(call.function.arguments)
        )
        
        messages.append({
            "role": "tool",
            "content": result.value,
            "name": call.function.name,
            "tool_call_id": call.id
        })
        
        # Get final answer
        final = openai_client.chat.completions.create(
            model=LLM_ENDPOINT_NAME,
            messages=messages,
            tools=tools
        )
        messages.append(final.choices[0].message.to_dict())
    
    return messages[-1].get('content', 'No response')

In [0]:
# Demo: Agent autonomously deciding to use tools
question = "How many host cities will the 2026 FIFA World Cup have, list all the stadiums?"
print(f"Question: {question}\n")

answer = run_agent(question)
print(f"Answer: {answer}")